In [1]:
import psycopg2, time, hashlib

conn = psycopg2.connect(
    host="localhost", port=5432,
    dbname="flightdb", user="postgres", password="1234"
)
conn.autocommit = True
cur = conn.cursor()

full_path = 'C:/Users/alber/Downloads/archive (1)/itineraries.csv'
csv_path_pg = 'C:/Users/alber/Downloads/archive (1)/itineraries_sample_pg.csv'

In [2]:
cur.execute("""
    DROP TABLE IF EXISTS flights_full;
    CREATE TABLE flights_full (
      "legId" VARCHAR(64), "searchDate" DATE, "flightDate" DATE,
      "startingAirport" CHAR(3), "destinationAirport" CHAR(3),
      "fareBasisCode" VARCHAR(32), "travelDuration" VARCHAR(16),
      "elapsedDays" INT, "isBasicEconomy" BOOLEAN, "isRefundable" BOOLEAN,
      "isNonStop" BOOLEAN, "baseFare" DECIMAL(10,2), "totalFare" DECIMAL(10,2),
      "seatsRemaining" INT, "totalTravelDistance" INT,
      "segmentsDepartureTimeEpochSeconds" TEXT, "segmentsDepartureTimeRaw" TEXT,
      "segmentsArrivalTimeEpochSeconds" TEXT, "segmentsArrivalTimeRaw" TEXT,
      "segmentsArrivalAirportCode" TEXT, "segmentsDepartureAirportCode" TEXT,
      "segmentsAirlineName" TEXT, "segmentsAirlineCode" TEXT,
      "segmentsEquipmentDescription" TEXT, "segmentsDurationInSeconds" TEXT,
      "segmentsDistance" TEXT, "segmentsCabinCode" TEXT
    );
""")

In [3]:
t0 = time.time()
with open(full_path, 'r', encoding='utf-8') as f:
    cur.copy_expert("COPY flights_full FROM STDIN WITH (FORMAT csv, HEADER true)", f)
elapsed = time.time() - t0
print(f"Full table load: {elapsed:.4f}s")

Full table load: 1870.9896s


In [4]:
t0 = time.time()
cur.execute("""
    DROP TABLE IF EXISTS flights;
    CREATE TABLE flights AS
    SELECT t.* FROM (
        SELECT *, ROW_NUMBER() OVER () AS rn FROM flights_full
    ) t
    WHERE rn % 2 = 0;
    ALTER TABLE flights DROP COLUMN rn;
""")
elapsed = time.time() - t0
print(f"Sample table build: {elapsed:.4f}s")

Sample table build: 1678.6797s


In [5]:
t0 = time.time()
with open(csv_path_pg, 'w', encoding='utf-8', newline='') as f:
    cur.copy_expert("COPY flights TO STDOUT WITH (FORMAT csv, HEADER true)", f)
elapsed = time.time() - t0
print(f"Export to CSV: {elapsed:.4f}s")

Export to CSV: 947.3905s


In [6]:
cur.execute("DROP TABLE IF EXISTS flights_full")

In [3]:
t0 = time.time()
cur.execute("SELECT COUNT(*) FROM flights")
n = cur.fetchone()[0]
print(f"Rows in flights table: {n}", f"{time.time()-t0:.4f}s")

Rows in flights table: 41069377 28.6276s


In [8]:
t0 = time.time()
cur.execute("""
    ALTER TABLE flights ADD COLUMN id BIGSERIAL PRIMARY KEY;
    CREATE INDEX idx_legid ON flights ("legId");
    CREATE INDEX idx_route_date ON flights ("startingAirport", "destinationAirport", "flightDate");
    CREATE INDEX idx_fare ON flights ("totalFare");
""")
elapsed = time.time() - t0
print(f"Index build: {elapsed:.4f}s")

Index build: 1090.5264s


In [3]:
t0 = time.time()
cur.execute('SELECT * FROM flights WHERE "legId" = %s', ('9335fae376c38bb61263281779f469ec',))
result = cur.fetchall()
print(len(result), "rows", f"{time.time()-t0:.4f}s")

1 rows 0.0098s


In [10]:
t0 = time.time()
cur.execute("""
    SELECT * FROM flights
    WHERE "startingAirport"='JFK' AND "destinationAirport"='LAX'
      AND "flightDate" BETWEEN '2022-06-01' AND '2022-06-30'
      AND "totalFare" < 300
""")
result = cur.fetchall()
print(len(result), "rows", f"{time.time()-t0:.4f}s")

3445 rows 0.9214s


In [11]:
t0 = time.time()
cur.execute("""
    SELECT "startingAirport", "destinationAirport",
           TO_CHAR("flightDate", 'YYYY-MM') AS month,
           AVG("totalFare") AS avg_fare, COUNT(*) AS n
    FROM flights
    GROUP BY "startingAirport", "destinationAirport", month
    ORDER BY month
""")
result = cur.fetchall()
print(len(result), "rows", f"{time.time()-t0:.4f}s")

1873 rows 240.2679s


In [12]:
t0 = time.time()
cur.execute("""
    SELECT "startingAirport", "destinationAirport", "isNonStop",
           AVG("totalFare") AS avg_fare
    FROM flights
    GROUP BY "startingAirport", "destinationAirport", "isNonStop"
""")
result = cur.fetchall()
print(len(result), "rows", f"{time.time()-t0:.4f}s")

450 rows 23.2191s


In [13]:
t0 = time.time()
cur.execute("""
    INSERT INTO flights ("legId", "searchDate", "flightDate", "startingAirport", "destinationAirport",
      "fareBasisCode", "travelDuration", "elapsedDays", "isBasicEconomy", "isRefundable", "isNonStop",
      "baseFare", "totalFare", "seatsRemaining", "totalTravelDistance",
      "segmentsDepartureTimeEpochSeconds", "segmentsDepartureTimeRaw",
      "segmentsArrivalTimeEpochSeconds", "segmentsArrivalTimeRaw",
      "segmentsArrivalAirportCode", "segmentsDepartureAirportCode",
      "segmentsAirlineName", "segmentsAirlineCode", "segmentsEquipmentDescription",
      "segmentsDurationInSeconds", "segmentsDistance", "segmentsCabinCode")
    VALUES ('synthetic-tier4-test', '2022-08-01', '2022-08-02', 'JFK', 'LAX',
      'TESTCODE', 'PT6H', 0, false, false, true, 300.00, 350.00, 10, 2475,
      '1658812800', '2022-08-02T08:00:00.000-04:00',
      '1658827200', '2022-08-02T12:00:00.000-04:00',
      'LAX', 'JFK', 'Delta', 'DL',
      'Boeing 737-800', '14400', '2475', 'coach')
""")
print(f"{time.time()-t0:.4f}s")

0.0206s


In [14]:
t0 = time.time()
cur.execute("""
    UPDATE flights SET "seatsRemaining" = "seatsRemaining" - 1
    WHERE "flightDate" = '2022-07-04' AND "startingAirport" = 'ATL'
""")
print(f"{time.time()-t0:.4f}s")

3.4288s
